# Cifar10 classification tricks

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fpschill/cas_main_dl_2026/blob/main/05_cifar10_training.ipynb)

In this notebook you will download the cifar10 dataset which contains quite small images (32x32x3) of 10 classes. The data is from the Canadian Institute For Advanced Research. You will plot examples of the images with the class label. Note that because the images are so small it is not always very easy to recognise which of the ten classes is on the image, even as a human. After loading the dataset you will train multiple models and compare the performances of the models on the testset.

**Dataset:**  You work with the Cifar10 dataset. You have 60'000 32x32 pixel color images of 10 classes ("airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck")

**Content:**
* load the original cifar10 data create a train, val and test dataset
* visualize samples of cifar10 dataset

* train a random forest on the pixel values
* train a cnn from scratch without normalization
* train a cnn from scratch with normalization
* train a cnn from scratch with dropout
* train a cnn from scratch with batchnorm
* train a cnn from scratch with data augmentation

* compare the performances of the models

* 🔧  your task at the end of the notebook: try to beat the models from this notebook


#### Imports

In the next two cells, we load all the required libraries and define some helper functions.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"  # must be set BEFORE keras is imported
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use("default")
from sklearn.metrics import confusion_matrix
import torch
import keras

print("Keras version:", keras.__version__)
print("Keras backend:", keras.backend.backend())
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

# Keras building blocks
from keras.models import Sequential
from keras.layers import Input, Dense, Conv2D, MaxPooling2D, Flatten, Activation, Dropout, BatchNormalization

# dataloader and augmentation strategy
from torch.utils.data import DataLoader
from torchvision import transforms as tr

### Helper functions

In [ ]:
class NumpyDataset(torch.utils.data.Dataset):
    def __init__(self, data, labels, transform=None):
        self.data = data
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image = self.data[idx]
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)
        return image, label



def plot_history(history):
  # plot the development of the accuracy and loss during training
  plt.figure(figsize=(12,4))
  plt.subplot(1,2,(1))
  plt.plot(history.history['accuracy'],linestyle='-.')
  plt.plot(history.history['val_accuracy'])
  plt.title('model accuracy')
  plt.ylabel('accuracy')
  plt.ylim(bottom=0)
  plt.xlabel('epoch')
  plt.legend(['train', 'valid'], loc='lower right')
  plt.subplot(1,2,(2))
  plt.plot(history.history['loss'],linestyle='-.')
  plt.plot(history.history['val_loss'])
  plt.title('model loss')
  plt.ylabel('loss')
  plt.ylim(bottom=0)
  plt.xlabel('epoch')
  plt.legend(['train', 'valid'], loc='upper right')


def reset_model(model):
  # return a fresh copy of the model: same architecture, loss and optimizer settings,
  # but new random weights and a new optimizer state.
  # Call it before model.fit, so that re-running the training cell starts from scratch
  # instead of continuing to train the already trained model.
  fresh = keras.models.clone_model(model)
  fresh.compile(loss=model.loss,
                optimizer=model.optimizer.__class__.from_config(model.optimizer.get_config()),
                metrics=['accuracy'])
  return fresh


def test_accuracy(model, dataloader):
  # fraction of correctly classified images (the dataloader must not be shuffled)
  pred = np.argmax(model.predict(dataloader, verbose=0), axis=1)
  return np.mean(pred == dataloader.dataset.labels)


# test accuracy of all models, filled in the sections below
results = {}


### Load and plot the data

In the next cell you will load the Cifar10 dataset, 50'000 images are in the training set and 10'000 are in the test dataset. You will use 10'000 images each for the train and validation dataset.
The labels are the class indices 0...9 (we use the loss `sparse_categorical_crossentropy`, so no one-hot encoding is needed).
You will plot one random example of each label and will see that the images are really small.

**NOTE** On the first run, the dataset will be downloaded (then cached). This might take 30 minutes due to the slow server.


In [ ]:
from keras.datasets import cifar10
(x_train_full, y_train_full), (X_test, y_test) = cifar10.load_data()

In [ ]:
# separate train, val and test dataset
# the labels are the class indices 0...9, flatten() turns their shape (N, 1) into (N,)
X_train, y_train = x_train_full[0:10000], y_train_full[0:10000].flatten()
X_val, y_val = x_train_full[20000:30000], y_train_full[20000:30000].flatten()
y_test = y_test.flatten()

print("train:", X_train.shape, y_train.shape)
print("val:  ", X_val.shape, y_val.shape)
print("test: ", X_test.shape, y_test.shape)

In [ ]:
class_names = np.array(["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"])
# sample image of each class
plt.figure(figsize=(15,15))
for i in range(len(class_names)):
    idx = np.random.choice(np.where(y_train == i)[0])
    plt.subplot(1,10,i+1)
    plt.imshow(X_train[idx])
    plt.title(f"{class_names[i]} {i}")

### Datasets and DataLoaders
The CNNs are trained with PyTorch DataLoaders, which deliver the images in batches of 64. The function `make_dataloaders` creates the DataLoaders for the train, validation and test data. With `transform` you choose how the images are preprocessed:
- without a transformation, the network gets the raw pixel values 0...255
- `scale_to_01` scales the pixel values to the range 0...1 (normalization)

With `train_transform` you can use a different transformation for the training data only. We will use this for the data augmentation.

In [ ]:
# scale the pixel values from 0...255 to 0...1
scale_to_01 = tr.Compose([
    tr.ToPILImage(),                          # convert numpy array to PIL image
    tr.ToTensor(),                            # convert PIL image to tensor (C, H, W), values in 0...1
    tr.Lambda(lambda x: x.permute(1, 2, 0))   # (C, H, W) -> (H, W, C), the format keras expects
])


def make_dataloaders(transform=None, train_transform=None, batch_size=64):
  # train_transform is only used for the training data, by default it is the same as transform
  if train_transform is None:
    train_transform = transform
  train_dataloader = DataLoader(NumpyDataset(X_train, y_train, train_transform), batch_size=batch_size, shuffle=True)
  val_dataloader = DataLoader(NumpyDataset(X_val, y_val, transform), batch_size=batch_size, shuffle=False)
  test_dataloader = DataLoader(NumpyDataset(X_test, y_test, transform), batch_size=batch_size, shuffle=False)

  images, targets = next(iter(train_dataloader))
  print(f'minimum value {images.min()} | maximum value {images.max()}')
  return train_dataloader, val_dataloader, test_dataloader

### RF on pixel values
In this section you will train a random forest on the raw pixel values of the images.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
clf = RandomForestClassifier(n_estimators=40,random_state=22)
clf.fit(X_train.reshape(len(X_train),32*32*3), y_train)

In [ ]:
pred = clf.predict(X_test.reshape(len(X_test),32*32*3))
results['rf on pixelvalues'] = np.mean(pred == y_test)
pd.DataFrame({'Acc': results})

### CNN from scratch without normalization
In this section you train a cnn from scratch to learn to classify the images into the right label. Normalization is not applied to the data. Note: with raw pixel values between 0 and 255, training occasionally gets stuck at the start (loss stays around 14.5, accuracy at chance level, 10%). If this happens, simply run the training cell again (it starts with new random weights).

We define this cnn in a function `build_cnn`, because we use the same network again in later sections.

In [ ]:
def build_cnn(learning_rate):
    model = Sequential()

    model.add(Input(shape=(32,32,3)))
    model.add(Conv2D(16,(3,3),activation="relu",padding="same"))
    model.add(Conv2D(16,(3,3),activation="relu",padding="same"))
    model.add(MaxPooling2D((2,2)))

    model.add(Conv2D(32,(3,3),activation="relu",padding="same"))
    model.add(Conv2D(32,(3,3),activation="relu",padding="same"))
    model.add(MaxPooling2D((2,2)))

    model.add(Flatten())
    model.add(Dense(500))
    model.add(Activation('relu'))
    model.add(Dense(300))
    model.add(Activation('relu'))
    model.add(Dense(100))
    model.add(Activation('relu'))

    model.add(Dense(10))
    model.add(Activation('softmax'))

    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=['accuracy'])
    return model


model = build_cnn(learning_rate=5e-5)
model.summary()

In [ ]:
train_dataloader, val_dataloader, test_dataloader = make_dataloaders()  # raw pixel values 0...255

In [ ]:
model = reset_model(model)  # start from fresh weights, also when re-running this cell
history = model.fit(train_dataloader, epochs=10, validation_data=val_dataloader, shuffle=False)

In [ ]:
plot_history(history)

In [ ]:
results['cnn from scratch without normalization'] = test_accuracy(model, test_dataloader)
pd.DataFrame({'Acc': results})

### CNN from scratch with normalization
In this section you train a cnn from scratch to learn to classify the images into the right label. Normalization is applied to the data.

**Learning rate:** in this section we use a learning rate of 5e-4, ten times higher than the 5e-5 in the section without normalization. The learning rate has to fit the scale of the input data: Adam changes every weight by roughly the learning rate in each step, and the effect of a weight change on the output of a layer is proportional to its input (Δw · x). With raw pixel values up to 255, the same step changes the output of the first layer roughly 255 times more than with values between 0 and 1. Because the network only uses ReLU activations, this larger scale carries through all layers. So without normalization, 5e-5 is already a large step, while with normalized data 5e-5 would learn very slowly. With normalized data, the network trains stably with the larger learning rate of 5e-4.

This also explains why the training accuracy can rise quickly without normalization: the network effectively trains with a large learning rate and memorizes the 10'000 training images. To compare the models, look at the validation (and test) accuracy, not the training accuracy.

In [ ]:
model = build_cnn(learning_rate=5e-4)  # same cnn as in the previous section

In [ ]:
train_dataloader, val_dataloader, test_dataloader = make_dataloaders(transform=scale_to_01)  # pixel values scaled to 0...1

In [ ]:
model = reset_model(model)  # start from fresh weights, also when re-running this cell
history = model.fit(train_dataloader, epochs=10, validation_data=val_dataloader, shuffle=False)

In [ ]:
plot_history(history)

In [ ]:
results['cnn from scratch with normalization'] = test_accuracy(model, test_dataloader)
pd.DataFrame({'Acc': results})

### CNN from scratch with Dropout
In this section you train a cnn from scratch to learn to classify the images into the right label. This time you will use dropout layers in the classification part. As in the previous section, the pixel values are scaled to the 0...1 range (on raw 0...255 values this model sometimes gets stuck at chance level).

In [ ]:
model  =  Sequential()

model.add(Input(shape=(32,32,3)))
model.add(Conv2D(16,(3,3),activation="relu",padding="same"))
model.add(Conv2D(16,(3,3),activation="relu",padding="same"))
model.add(MaxPooling2D((2,2)))

model.add(Conv2D(32,(3,3),activation="relu",padding="same"))
model.add(Conv2D(32,(3,3),activation="relu",padding="same"))
model.add(MaxPooling2D((2,2)))

model.add(Flatten())
model.add(Dropout(0.3))   #<------ Dropout layers
model.add(Dense(500))
model.add(Activation('relu'))
model.add(Dropout(0.3))   #<------ Dropout layers
model.add(Dense(300))
model.add(Activation('relu'))
model.add(Dropout(0.3))   #<------ Dropout layers
model.add(Dense(100))
model.add(Activation('relu'))

model.add(Dense(10))
model.add(Activation('softmax'))

model.compile(loss='sparse_categorical_crossentropy',
              optimizer=keras.optimizers.Adam(learning_rate=5e-4),
              metrics=['accuracy'])

model.summary()

In [ ]:
train_dataloader, val_dataloader, test_dataloader = make_dataloaders(transform=scale_to_01)  # pixel values scaled to 0...1

In [ ]:
model = reset_model(model)  # start from fresh weights, also when re-running this cell
history = model.fit(train_dataloader, epochs=15, validation_data=val_dataloader, shuffle=False)

In [ ]:
plot_history(history)

In [ ]:
results['cnn from scratch with dropout'] = test_accuracy(model, test_dataloader)
pd.DataFrame({'Acc': results})

### CNN from scratch with Batchnorm
In this section you train a cnn from scratch to learn to classify the images into the right label. This time you will use batchnorm on the input and in the convolutional part of the network. Note that we use the original images and do not normalize them. Because the batchnorm layer on the input normalizes the data inside the network, we use the same learning rate of 5e-4 as for the normalized data (see the note in the section with normalization).

In [ ]:
model  =  Sequential()

model.add(Input(shape=(32,32,3)))
model.add(BatchNormalization())                    #<------ Batchnorm on the input
model.add(Conv2D(16,(3,3),padding="same"))
model.add(BatchNormalization())                    #<------ Batchnorm layers
model.add(Activation('relu'))
model.add(Conv2D(16,(3,3),padding="same"))
model.add(BatchNormalization())                    #<------ Batchnorm layers
model.add(Activation('relu'))
model.add(MaxPooling2D((2,2)))

model.add(Conv2D(32,(3,3),padding="same"))
model.add(BatchNormalization())                    #<------ Batchnorm layers
model.add(Activation('relu'))
model.add(Conv2D(32,(3,3),padding="same"))
model.add(BatchNormalization())                    #<------ Batchnorm layers
model.add(Activation('relu'))
model.add(MaxPooling2D((2,2)))

model.add(Flatten())
model.add(Dense(500))
model.add(Activation('relu'))
model.add(Dense(300))
model.add(Activation('relu'))
model.add(Dense(100))
model.add(Activation('relu'))

model.add(Dense(10))
model.add(Activation('softmax'))

model.compile(loss='sparse_categorical_crossentropy',
              optimizer=keras.optimizers.Adam(learning_rate=5e-4),
              metrics=['accuracy'])

model.summary()

In [ ]:
train_dataloader, val_dataloader, test_dataloader = make_dataloaders()  # raw pixel values 0...255, the batchnorm layer on the input normalizes them

In [ ]:
model = reset_model(model)  # start from fresh weights, also when re-running this cell
history = model.fit(train_dataloader, epochs=15, validation_data=val_dataloader, shuffle=False)

In [ ]:
plot_history(history)

**Why does the validation loss increase?** In the plot above, the training loss keeps going down and the training accuracy approaches 100%, but after a few epochs the validation loss starts to rise again while the validation accuracy stays roughly constant. This is **overfitting**, not an unstable training:

- Batchnorm makes the optimization much faster, so the network can memorize the 10'000 training images within a few epochs. The dense part of the network has more than 1 million parameters, and there is nothing (like dropout or data augmentation) that stops it from memorizing.
- The accuracy only checks whether the class with the highest probability is correct. The cross-entropy loss also measures how *confident* the prediction is. While the network memorizes the training data, its predictions get more and more confident, also for the validation images it classifies wrongly. A confident wrong prediction (probability close to 0 for the true class) gives a large loss, $-\log(p)$. So the validation loss rises, even though about the same number of validation images are classified correctly.

Batchnorm therefore helps to train faster, but on its own it does not make the network generalize better. To fight overfitting you can combine it with dropout or data augmentation, train for fewer epochs, or use early stopping: `keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)` stops the training when the validation loss no longer improves and keeps the weights of the best epoch.

In [ ]:
results['cnn from scratch with batchnorm'] = test_accuracy(model, test_dataloader)
pd.DataFrame({'Acc': results})

#### Exercise
Calculate the confusion matrix of the networks.  
Play around with the dropout rate and the position of the batchnorm.

### CNN from scratch with Data Augmentation
In this section you train a cnn from scratch to learn to classify the images into the right label. This time you will use data augmentation, so the network will train on slightly different versions of the images in each epoch. As in the section with normalization, the pixel values are scaled to the 0...1 range (training on raw 0...255 values with this model sometimes gets stuck at chance level).

Data Augmentation is especially helpful if you do not have lots of data. Another approach to train with few data is to use a pretrained neural network. This will be covered in the next notebook.

In [ ]:
model = build_cnn(learning_rate=5e-4)  # same cnn as in the sections without and with normalization

For the augmentation we use a different transformation for the training data: in addition to the scaling to 0...1, each image is randomly flipped, rotated and cropped. The validation and test data are only scaled, not augmented.

In [ ]:
# transformation for the training data: scaling to 0...1 plus random augmentation
augment_and_scale = tr.Compose([
    tr.ToPILImage(),                          # convert numpy array to PIL image
    tr.ToTensor(),                            # convert PIL image to tensor, values in 0...1
    tr.RandomHorizontalFlip(),                # random horizontal flip
    tr.RandomRotation(15),                    # random rotation
    tr.RandomCrop(32, padding=4),             # random cropping with padding
    tr.Lambda(lambda x: x.permute(1, 2, 0))   # (C, H, W) -> (H, W, C)
])

train_dataloader, val_dataloader, test_dataloader = make_dataloaders(transform=scale_to_01, train_transform=augment_and_scale)

- see that the image is now randomly shifted cropped and flipped, each time the images enters the training loop, this operation is happening on the fly while training the model

In [ ]:
images, targets = next(iter(train_dataloader))
plt.figure(figsize=(2,2))
plt.imshow(images[0].numpy())
plt.show()

In [ ]:
model = reset_model(model)  # start from fresh weights, also when re-running this cell
history = model.fit(train_dataloader, epochs=30, validation_data=val_dataloader, shuffle=False)

In [ ]:
plot_history(history)

In [ ]:
results['cnn from scratch with data augmentation'] = test_accuracy(model, test_dataloader)
pd.DataFrame({'Acc': results})

## 🔧 **YOUR TASK:**
- Try to beat the performance of the best network with your own neural network.  
- you might want to combine some approaches from above
- don't forget to normalize also the testset if you use normalization (which you should use anyway😉)
Further:
- use larger training dataset (not just 10000)
- train for more epochs

In [ ]:
### YOUR CODE ###